In [2]:
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score


In [7]:
# =========================
# Load data
# =========================
df = pd.read_csv("abstention_sft_gemma_xml.csv")

# Normalize answers
df["final_answer"] = df["final_answer"].astype(str).str.lower().str.strip()
df["pred_final_answer"] = df["pred_final_answer"].astype(str).str.lower().str.strip()

# Fill reasoning
df["reasoning"] = df["reasoning"].fillna("").astype(str)
df["pred_reasoning"] = df["pred_reasoning"].fillna("").astype(str)


# =========================
# Exact Match Accuracy
# =========================
accuracy = accuracy_score(df["final_answer"], df["pred_final_answer"])


# =========================
#  Binary Accuracy (paper metric)
# =========================
def is_unanswerable(ans):
    return ans == "idk"

df["gt_binary"] = df["final_answer"].apply(lambda x: 0 if is_unanswerable(x) else 1)
df["pred_binary"] = df["pred_final_answer"].apply(lambda x: 0 if is_unanswerable(x) else 1)

binary_accuracy = accuracy_score(df["gt_binary"], df["pred_binary"])


# =========================
# BLEU / ROUGE / Token-F1
# =========================
smooth = SmoothingFunction().method1

rouge_scorer_obj = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def compute_token_f1(ref, pred):
    ref_tokens = ref.split()
    pred_tokens = pred.split()

    if len(ref_tokens) == 0 or len(pred_tokens) == 0:
        return 0.0

    common = set(ref_tokens) & set(pred_tokens)

    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(ref_tokens)

    return 2 * (precision * recall) / (precision + recall)


bleu_list = []
rouge1_list = []
rouge2_list = []
rougeL_list = []
token_f1_list = []

for ref, pred in zip(df["reasoning"], df["pred_reasoning"]):

    ref_tokens = ref.split()
    pred_tokens = pred.split()

    if len(pred_tokens) == 0:
        bleu = 0.0
    else:
        bleu = sentence_bleu(
            [ref_tokens],
            pred_tokens,
            smoothing_function=smooth
        )

    bleu_list.append(bleu)

    scores = rouge_scorer_obj.score(ref, pred)
    rouge1_list.append(scores["rouge1"].fmeasure)
    rouge2_list.append(scores["rouge2"].fmeasure)
    rougeL_list.append(scores["rougeL"].fmeasure)

    token_f1_list.append(compute_token_f1(ref, pred))


# =========================
# BERTScore
# =========================
P, R, F1 = bert_score(
    df["pred_reasoning"].tolist(),
    df["reasoning"].tolist(),
    lang="en",
    verbose=True
)


# =========================
# Save per-row metrics
# =========================
df["bleu"] = bleu_list
df["rouge1"] = rouge1_list
df["rouge2"] = rouge2_list
df["rougeL"] = rougeL_list
df["token_f1"] = token_f1_list
df["bertscore_precision"] = P.tolist()
df["bertscore_recall"] = R.tolist()
df["bertscore_f1"] = F1.tolist()

df["correct"] = (df["final_answer"] == df["pred_final_answer"]).astype(int)
df["binary_correct"] = (df["gt_binary"] == df["pred_binary"]).astype(int)


# =========================
# Averages
# =========================
avg_bleu = np.mean(bleu_list)
avg_rouge1 = np.mean(rouge1_list)
avg_rouge2 = np.mean(rouge2_list)
avg_rougeL = np.mean(rougeL_list)
avg_token_f1 = np.mean(token_f1_list)
avg_bertscore_f1 = F1.mean().item()


# =========================
# Print results
# =========================
print("\n===== OVERALL METRICS =====")
print(f"Exact Accuracy   : {accuracy:.4f}")
print(f"Binary Accuracy  : {binary_accuracy:.4f}")  # paper metric
print(f"BLEU             : {avg_bleu:.4f}")
print(f"ROUGE-1          : {avg_rouge1:.4f}")
print(f"ROUGE-2          : {avg_rouge2:.4f}")
print(f"ROUGE-L          : {avg_rougeL:.4f}")
print(f"BERTScore F1     : {avg_bertscore_f1:.4f}")
print(f"Token F1         : {avg_token_f1:.4f}")


# =========================
# Save CSV
# =========================
df.to_csv("evaluation_sft_gemma.csv", index=False)

print("\nSaved detailed metrics to evaluation_gemma.csv")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


AcceleratorError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [5]:
# ==========================================================
# FULL WORKING EVALUATION CODE
# Safe for GPU OOM (BERTScore on CPU)
# ==========================================================

import pandas as pd
import numpy as np
import torch
import gc

from sklearn.metrics import accuracy_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score

# =========================
# OPTIONAL: FREE GPU MEMORY
# =========================
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("abstention_sft_gemma_xml.csv")

# =========================
# NORMALIZE TEXT
# =========================
df["final_answer"] = df["final_answer"].astype(str).str.lower().str.strip()
df["pred_final_answer"] = df["pred_final_answer"].astype(str).str.lower().str.strip()

df["reasoning"] = df["reasoning"].fillna("").astype(str)
df["pred_reasoning"] = df["pred_reasoning"].fillna("").astype(str)

# =========================
# EXACT ACCURACY
# =========================
accuracy = accuracy_score(
    df["final_answer"],
    df["pred_final_answer"]
)

# =========================
# BINARY ACCURACY
# =========================
def is_unanswerable(ans):
    return ans == "idk"

df["gt_binary"] = df["final_answer"].apply(
    lambda x: 0 if is_unanswerable(x) else 1
)

df["pred_binary"] = df["pred_final_answer"].apply(
    lambda x: 0 if is_unanswerable(x) else 1
)

binary_accuracy = accuracy_score(
    df["gt_binary"],
    df["pred_binary"]
)

# =========================
# BLEU / ROUGE / TOKEN-F1
# =========================
smooth = SmoothingFunction().method1

rouge_scorer_obj = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def compute_token_f1(ref, pred):

    ref_tokens = ref.split()
    pred_tokens = pred.split()

    if len(ref_tokens) == 0 or len(pred_tokens) == 0:
        return 0.0

    common = set(ref_tokens) & set(pred_tokens)

    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)

bleu_list = []
rouge1_list = []
rouge2_list = []
rougeL_list = []
token_f1_list = []

for ref, pred in zip(df["reasoning"], df["pred_reasoning"]):

    ref_tokens = ref.split()
    pred_tokens = pred.split()

    # BLEU
    if len(pred_tokens) == 0:
        bleu = 0.0
    else:
        bleu = sentence_bleu(
            [ref_tokens],
            pred_tokens,
            smoothing_function=smooth
        )

    bleu_list.append(bleu)

    # ROUGE
    scores = rouge_scorer_obj.score(ref, pred)

    rouge1_list.append(scores["rouge1"].fmeasure)
    rouge2_list.append(scores["rouge2"].fmeasure)
    rougeL_list.append(scores["rougeL"].fmeasure)

    # Token F1
    token_f1_list.append(
        compute_token_f1(ref, pred)
    )

# =========================
# BERTScore (CPU SAFE)
# =========================
P, R, F1 = bert_score(
    df["pred_reasoning"].tolist(),
    df["reasoning"].tolist(),
    lang="en",
    verbose=True,
    device="cpu"
)

# =========================
# SAVE PER-ROW METRICS
# =========================
df["bleu"] = bleu_list
df["rouge1"] = rouge1_list
df["rouge2"] = rouge2_list
df["rougeL"] = rougeL_list
df["token_f1"] = token_f1_list

df["bertscore_precision"] = P.tolist()
df["bertscore_recall"] = R.tolist()
df["bertscore_f1"] = F1.tolist()

df["correct"] = (
    df["final_answer"] == df["pred_final_answer"]
).astype(int)

df["binary_correct"] = (
    df["gt_binary"] == df["pred_binary"]
).astype(int)

# =========================
# OVERALL AVERAGES
# =========================
avg_bleu = np.mean(bleu_list)
avg_rouge1 = np.mean(rouge1_list)
avg_rouge2 = np.mean(rouge2_list)
avg_rougeL = np.mean(rougeL_list)
avg_token_f1 = np.mean(token_f1_list)
avg_bertscore_f1 = F1.mean().item()

# =========================
# PRINT RESULTS
# =========================
print("\n===== OVERALL METRICS =====")
print(f"Exact Accuracy   : {accuracy:.4f}")
print(f"Binary Accuracy  : {binary_accuracy:.4f}")
print(f"BLEU             : {avg_bleu:.4f}")
print(f"ROUGE-1          : {avg_rouge1:.4f}")
print(f"ROUGE-2          : {avg_rouge2:.4f}")
print(f"ROUGE-L          : {avg_rougeL:.4f}")
print(f"BERTScore F1     : {avg_bertscore_f1:.4f}")
print(f"Token F1         : {avg_token_f1:.4f}")

# =========================
# SAVE CSV
# =========================
output_eval = "evaluation_sft_gemma.csv"

df.to_csv(output_eval, index=False)

print(f"\nSaved detailed metrics to {output_eval}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/107 [00:00<?, ?it/s]

KeyboardInterrupt: 